# Summit 2026 HOL — End-to-End Validation
## Module 08: Full Scenario Test + DORA Completion Check

_Icons used throughout this lab: 🛠️ Action  📌 Note  🔹 Info_

---

### What This Module Does:

1. 🛠️ Runs a **full end-to-end scenario** across all components you built
2. ✅ Validates that every object exists and is functional
3. 🛠️ Runs the **DORA completion check** to confirm lab completion
4. 📌 Provides cleanup instructions

---

### Estimated Time: **10–15 minutes**

### Prerequisites:
- Modules 01–07 complete

## Step 1: Set Context

In [ ]:
%%sql -r SetContext
USE ROLE NEXUS_ADMIN;
USE DATABASE NEXUS_HOL;
USE WAREHOUSE NEXUS_WH;

## Step 2: ✅ Validate All Objects Exist

### Confirm every component from Modules 01–07 is in place

In [ ]:
%%sql -r SematicViews
-- Check: Semantic View exists
SHOW SEMANTIC VIEWS IN SCHEMA NEXUS_HOL.SEMANTIC;

In [ ]:
%%sql -r SearchServices
-- Check: Cortex Search Service exists
SHOW CORTEX SEARCH SERVICES IN SCHEMA NEXUS_HOL.AGENTS;

In [ ]:
%%sql -r Agents
-- Check: Agent exists
SHOW AGENTS IN SCHEMA NEXUS_HOL.AGENTS;

In [ ]:
%%sql -r MCPConnectors
-- Check: MCP Connector exists
SHOW EXTERNAL MCP SERVERS IN SCHEMA NEXUS_HOL.AGENTS;

In [ ]:
%%sql -r CustomAIFunctions
-- Check: Custom AI Function exists
SHOW USER FUNCTIONS LIKE 'CLASSIFY_TRADE_CONVICTION' IN SCHEMA NEXUS_HOL.ANALYTICS;

In [ ]:
%%sql -r SIOnlyRole
-- Check: SI-only role exists
SHOW ROLES LIKE 'NEXUS_SI_USER';

### 📌 Expected Results:

| Check | Expected Object |
|-------|----------------|
| Semantic View | `NEXUS_CAPITAL_SV` |
| Cortex Search | `NEXUS_RESEARCH_SEARCH` |
| Agent | `NEXUS_AGENT` |
| MCP Server | `NEXUS_ATLASSIAN_MCP` (and optionally Gmail, Drive, Calendar) |
| Custom AI Function | `CLASSIFY_TRADE_CONVICTION` |
| SI Role | `NEXUS_SI_USER` |

If any are missing, go back to the relevant module and re-run the creation cell.

 ## Step 3: 🛠️ Full Scenario Test

### Run one question that exercises the full stack

Open **CoWork** (AI & ML → Cortex AI → CoWork → Nexus Capital Analyst) and ask:

> *"Which clients have the highest unrealized gains in technology stocks, and what does our research say about the tech sector 
outlook? Should we be taking profits?"*

### What to verify:

This single question proves the entire pipeline works:

| What Should Happen | Which Module Built It |
|---|---|
| Agent queries POSITIONS + CLIENTS for tech gains | Module 02 (Semantic View) + Module 03 (Agent) |
| Agent searches research notes for tech outlook | Module 02 (Cortex Search) + Module 03 (Agent) |
| Agent synthesizes both into a recommendation | Module 03 (Orchestration) |
| Response shows thinking steps + citations | Module 04 (SI) |
| You can save the chart as an artifact | Module 04 (Artifacts) |
| MCP connector is available if you want to create a ticket | Module 05 (MCP) |

If you get a complete answer with both structured data (client gains table) and unstructured insights (research team outlook) —
your full stack is working.

## Optional: 🛠️ DORA API Integration Setup

Run the following cell **only if you have not already created the DORA API integration, SE Greeter function, and SE Grader function in this Snowflake account**.

This setup helps avoid the common DORA issue where your Snowflake email is not tied to an account locator in Snowhouse. Before running, replace the placeholder values in the `se_greeting` call with your Snowflake email and name.

In [ ]:
-- Optional DORA API Integration Setup
-- Run only if the DORA / SE Greeter setup has not already been completed in this account.

USE ROLE ACCOUNTADMIN;

-- Create API integration
CREATE OR REPLACE API INTEGRATION dora_api_integration
  API_PROVIDER = AWS_API_GATEWAY
  API_AWS_ROLE_ARN = 'arn:aws:iam::321463406630:role/snowflakeLearnerAssumedRole'
  ENABLED = TRUE
  API_ALLOWED_PREFIXES = (
    'https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora'
  );

-- Confirm integration
SHOW INTEGRATIONS;

-- Create utility database
CREATE OR REPLACE DATABASE util_db;

-- Create greeting function
CREATE OR REPLACE EXTERNAL FUNCTION util_db.public.se_greeting(
    email VARCHAR,
    firstname VARCHAR,
    middlename VARCHAR,
    lastname VARCHAR
)
RETURNS VARIANT
API_INTEGRATION = dora_api_integration
CONTEXT_HEADERS = (
  CURRENT_TIMESTAMP,
  CURRENT_ACCOUNT,
  CURRENT_STATEMENT,
  CURRENT_ACCOUNT_NAME
)
AS 'https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora/greeting';

-- IMPORTANT: Replace these values with your Snowflake email and name before running.
-- Example:
-- SELECT util_db.public.se_greeting(
--   'diana.shaw@snowflake.com', 'Diana', '', 'Shaw');

SELECT util_db.public.se_greeting(
  '<your_email@snowflake.com>',
  '<Your First Name>',
  '<Your Middle Name or empty string>',
  '<Your Last Name>'
);

-- Create grading function
CREATE OR REPLACE EXTERNAL FUNCTION util_db.public.se_grader(
    step VARCHAR,
    passed BOOLEAN,
    actual INTEGER,
    expected INTEGER,
    description VARCHAR
)
RETURNS VARIANT
API_INTEGRATION = dora_api_integration
CONTEXT_HEADERS = (
  CURRENT_TIMESTAMP,
  CURRENT_ACCOUNT,
  CURRENT_STATEMENT,
  CURRENT_ACCOUNT_NAME
)
AS 'https://awy6hshxy4.execute-api.us-west-2.amazonaws.com/dev/edu_dora/grader';

## Step 4: 🛠️ DORA Completion Check

### Run this validation to confirm your lab is complete

This checks:
- Semantic view exists with dimensions and metrics defined
- Agent exists and has been called (at least 5 interactions)
- Cortex Search service is active
- MCP server is configured
- Custom AI Function is registered

In [ ]:
%%sql -r DORA_SL1_01
-- SL1_01: Validate Summit 2026 HOL — Build an AI-Powered Enterprise Application
-- Requires: NEXUS_AGENT created and used 5+ times (proves Modules 01-07 were completed)

SELECT util_db.public.se_grader(
    step,
    (actual >= expected),
    actual,
    expected,
    description
) AS graded_results
FROM (
    SELECT
        'SL1_01' AS step,
        (SELECT COUNT(*)
         FROM SNOWFLAKE.ACCOUNT_USAGE.CORTEX_AGENT_USAGE_HISTORY
         WHERE AGENT_NAME = 'NEXUS_AGENT') AS actual,
        5 AS expected,
        '✅ Summit 2026 HOL Complete: NEXUS_AGENT used 5+ times' AS description
);

### 📌 If the check fails:

The most common reason is not enough agent interactions. Go back to Module 04 (CoWork chat) or Module 03 (DATA_AGENT_RUN) and ask a few more questions. The DORA check requires at least **5 agent calls** total.

> **Note:** The `CORTEX_AGENT_USAGE_HISTORY` view may have up to a 45-minute latency. If you just ran queries, wait and re-run the check.

## Step 5: What You Built (Architecture Summary)

```
┌─────────────────────────────────────────────────────────────────┐
│                    NEXUS CAPITAL AI APPLICATION                 │
│                                                                 │
│  ┌───────────────┐  ┌──────────────────┐  ┌──────────────────┐  │
│  │ Gold Layer    │  │ Semantic Layer   │  │ Agent Layer      │  │
│  │ (Module 01)   │  │ (Module 02)      │  │ (Module 03)      │  │
│  │               │  │                  │  │                  │  │
│  │ • Clients     │→ │ • Semantic View  │→ │ • NEXUS_AGENT    │  │
│  │ • Positions   │  │ • Verified Qrys  │  │ • Analyst Tool   │  │
│  │ • Trades      │  │ • SQL Gen Rules  │  │ • Search Tool    │  │
│  │ • Metrics     │  │                  │  │ • Chart Tool     │  │
│  │ • Research    │→ │ • Cortex Search  │→ │                  │  │
│  └───────────────┘  └──────────────────┘  └────────┬─────────┘  │
│                                                    │            │
│  ┌─────────────────────────────────────────────────┼──────────┐ │
│  │                    DELIVERY                     │          │ │
│  │                                                 ▼          │ │
│  │  ┌──────────────┐  ┌───────────────┐  ┌─────────────────┐  │ │
│  │  │ CoWork Chat  │  │ REST API      │  │ MCP Connectors  │  │ │
│  │  │ (Module 04)  │  │ (Module 03)   │  │ (Module 05)     │  │ │
│  │  │ Business     │  │ Developers    │  │ Jira, Gmail,    │  │ │
│  │  │ Users        │  │ & Apps        │  │ Slack, Drive    │  │ │
│  │  └──────────────┘  └───────────────┘  └─────────────────┘  │ │
│  └────────────────────────────────────────────────────────────┘ │
│                                                                 │
│  ┌─────────────────────────────────────────────────────────────┐│
│  │                    GOVERNANCE (Module 06)                   ││
│  │  RBAC • Guardrails • PII Redaction • Credit Budgets • Audit ││
│  └─────────────────────────────────────────────────────────────┘│
│                                                                 │
│  ┌─────────────────────────────────────────────────────────────┐│
│  │                    DEVELOPER TOOLS (Module 07)              ││
│  │  CoCo • Query Optimization • Custom AI Functions            ││
│  └─────────────────────────────────────────────────────────────┘│
└─────────────────────────────────────────────────────────────────┘
```

## 📌 Cleanup (Optional)

### When you're done with the lab, run these to stop billing:

```sql
-- Suspend the warehouse (stops compute billing)
ALTER WAREHOUSE NEXUS_WH SUSPEND;

-- To fully remove all lab objects:
DROP DATABASE IF EXISTS NEXUS_HOL;
DROP WAREHOUSE IF EXISTS NEXUS_WH;
DROP ROLE IF EXISTS NEXUS_ADMIN;
DROP ROLE IF EXISTS NEXUS_SI_USER;
```

> **Keep it running?** If you plan to demo this to customers or use it for practice, just suspend the warehouse. The database and agent cost nothing when idle.

---

## ✅ Summit 2026 HOL Complete!

### What you built in ~3 hours:

| Module | What You Built |
|--------|---------------|
| 01 | Gold-layer analytics environment (pre-loaded data) |
| 02 | Semantic view + Cortex Search service |
| 03 | Cortex Agent (SQL + REST API + Python) |
| 04 | CoWork deployment (CoWork chat + CoWork-only role) |
| 05 | MCP Connectors (Atlassian + optional Google/Slack) |
| 06 | AI Security & Governance (RBAC, Guardrails, PII Redaction) |
| 07 | CoCo optimization + Custom AI Function |
| 08 | End-to-end validation |

---

### Optional Extensions (continue on your own):

| Module | What You'd Build |
|--------|------------------|
| 09 | Transactional Layer — Snowflake Postgres instance with live trades |
| 10 | Streaming & Transformation — Dynamic Tables for CDC into analytics |

---

### Key Resources:

| Resource | Link |
|----------|------|
| CoWork Documentation | [Docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/snowflake-intelligence) |
| Cortex Agents Documentation | [Docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents) |
| MCP Connectors | [Docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-agents-mcp-connectors) |
| CoCo | [Docs](https://docs.snowflake.com/en/user-guide/cortex-code/cortex-code) |
| Semantic Views | [Docs](https://docs.snowflake.com/en/user-guide/snowflake-cortex/cortex-analyst/semantic-view) |
| CoWork GA Hands-On Lab (full L200) | [GitLab](https://github.com/snowflake-corp/collegeai/blob/initial-import/SI_GA_HOL/README.md) |
| Ontology on Snowflake | [Scout](https://scout.snowflake.com/learn/courses/50217/ontology-on-snowflake-enabling-business-aware-ai) |

---

### Talking Points (Full Lab Summary):

- "In 3 hours, I built a production-ready AI application that answers questions from structured and unstructured data, takes action in enterprise systems, and is fully governed."
- "Same agent, three delivery channels: REST API for developers, CoWork for business users, MCP for enterprise automation."
- "Governance isn't bolted on — it's built in. RBAC, PII masking, guardrails, credit budgets, and full audit trail from day one."
- "The semantic view is the single source of truth for AI understanding. Build it once, and every agent, every user, every tool gets consistent answers."

---

**Questions?** Contact Diana Shaw or check the lab documentation.